# Модельный проект: Autoencoder (AE) на Fashion-MNIST

**Тема проекта:** Autoencoder (AE) — сжатие и восстановление изображений одежды.

**Почему AE?** Это идеальный учебный модельный проект:
- Быстро обучается (минуты на CPU, секунды на GPU).
- Даёт наглядные результаты: реконструкции + латентное пространство.
- Позволяет пройти полный цикл ML-проекта: постановка → данные → модель → эксперименты → ретроспектива.

**Цель семинара:** показать студентам, как устроен модельный проект: какие артефакты нужны, как ими управлять и как связать управление с кодом.

## 1. Project Charter (устав проекта)

Это **одна страница**, которая отвечает на вопросы: зачем мы это делаем, что получим, в каких рамках.

| Поле | Значение |
|------|----------|
| **Название** | AE для сжатия изображений одежды |
| **Спонсор / заказчик** | Учебный семинар по DL |
| **Проблема** | Нужен компактный способ представления 28×28 изображений для downstream-задач (кластеризация, поиск похожих) |
| **SMART-цель** | К концу семинара получить AE с reconstruction MSE < 0.02 на val Fashion-MNIST и латентным dim 2 для визуализации |
| **Входит в scope** | Vanilla AE, эксперименты с латентным размером [2, 8, 32], визуализация реконструкций и латентного пространства |
| **Не входит в scope** | VAE, conditional AE, обучение на CIFAR-10, деплой |
| **Стейкхолдеры** | Студенты, преподаватель, (условно) «аналитик данных» |
| **Критерий успеха** | Восстановленные изображения визуально похожи на оригинал; латентные кластеры для разных классов различимы |
| **Ограничения** | 1 час на семинар, CPU-only или слабый GPU, минимум зависимостей (torch, torchvision, sklearn) |

## 2. WBS (Work Breakdown Structure) и Roadmap

**WBS** — разбиение проекта на задачи, которые можно оценить и отслеживать.

```
1. Инициация
   1.1. Постановка цели и ограничений (charter)
   1.2. Согласование критериев успеха
2. Подготовка данных
   2.1. Загрузка Fashion-MNIST
   2.2. EDA: примеры, распределение классов
   2.3. Train/val/test split, нормализация
3. Моделирование
   3.1. Проектирование архитектуры (encoder/decoder)
   3.2. Реализация модели
   3.3. Цикл обучения + валидация
4. Эксперименты
   4.1. Сравнение latent_dim ∈ {2, 8, 32}
   4.2. Логирование в таблицу
   4.3. Визуализация реконструкций и латентного пространства
5. Завершение
   5.1. Сбор результатов в итоговую таблицу
   5.2. Ретроспектива
```

**Roadmap (упрощённый Gantt):**

| День | Задача | Owner | Статус |
|------|--------|-------|--------|
| 1 | Charter + WBS | PM/преподаватель | ✅ |
| 1 | Данные + EDA | Data | ✅ |
| 2 | Baseline AE | ML | ✅ |
| 2 | Эксперименты (3 конфигурации) | ML | ✅ |
| 3 | Визуализации + итоговая таблица | ML/PM | ✅ |
| 3 | Ретроспектива | Все | ✅ |

## 3. Kanban board и Risk register

**Kanban** — доска для отслеживания задач. В маркдауне её удобно имитировать чек-боксами.

### 📋 Backlog
- [ ] Добавить VAE (latent как распределение)
- [ ] Попробовать conditional AE
- [ ] Сравнить с PCA baseline

### 🚧 In Progress
- [x] Эксперименты с latent_dim
- [ ] Визуализация латентного пространства через t-SNE

### ✅ Done
- [x] Charter
- [x] WBS
- [x] Загрузка данных
- [x] Модель AE
- [x] Скрипт обучения

**Risk register:**

| Риск | Вероятность | Влияние | Митигация |
|------|-------------|---------|-----------|
| Переобучение на train | Средняя | Высокое | Валидация на каждой эпохе, ранняя остановка |
| Латентное dim=2 слишком мало | Высокая | Среднее | Сравнить с dim=8 и dim=32 |
| Медленное обучение на CPU | Средняя | Среднее | Уменьшить epochs, использовать подвыборку |
| Нестабильные потери | Низкая | Высокое | Adam + lr=1e-3, нормализация данных |
| Плоская визуализация (все в одной точке) | Средняя | Среднее | Проверить, что модель обучилась (loss < 0.05) |

## 4. Experiment log и Definition of Done

**Experiment log** — таблица, в которой фиксируем **каждый** запуск. В реальных проектах её ведут в MLflow, Weights & Biases, Sacred. Здесь — pandas-DataFrame для наглядности.

Столбцы:
- `run_id` — уникальный идентификатор запуска
- `config` — hyperparams (latent_dim, lr, epochs, batch_size)
- `metrics` — train_loss, val_loss, время
- `artifacts` — путь к чекпоинту, графику

**Definition of Done (DoD):**

- [ ] Модель обучается без ошибок, loss падает.
- [ ] На валидации MSE < 0.02 (для dim=32).
- [ ] Построены реконструкции оригинал vs восстановление.
- [ ] Построено латентное пространство (scatter для dim=2, PCA/t-SNE для dim=32).
- [ ] Заполнена итоговая таблица экспериментов.
- [ ] Записаны выводы (что получилось, что можно улучшить).

**Ретроспектива** — в конце ноутбука (что было хорошо, что можно улучшить, что делаем в следующий раз).

---
# 💻 Реализация

In [ ]:
import os, time, copy, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| Torch:", torch.__version__)

Device: cpu | Torch: 2.14.0


: 

In [ ]:
# Загрузка Fashion-MNIST (встроен в torchvision — стабильно работает)
transform = transforms.ToTensor()

train_ds = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=256,        shuffle=False, num_workers=0)

class_names = ['T-shirt/top','Trouser','Pullover','Dress','Coat','Sandal','Shirt','Sneaker','Bag','Ankle boot']
print("Train:", len(train_ds), "Test:", len(test_ds))

100%|██████████| 26.4M/26.4M [00:06<00:00, 3.79MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 630kB/s]
100%|██████████| 4.42M/4.42M [00:06<00:00, 707kB/s] 
100%|██████████| 5.15k/5.15k [00:00<00:00, 18.4MB/s]

In [ ]:
# EDA: примеры по классам
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls_idx, ax in enumerate(axes.ravel()):
    idx = np.where(np.array(train_ds.targets) == cls_idx)[0][0]
    img, _ = train_ds[idx]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(class_names[cls_idx], fontsize=9)
    ax.axis("off")
plt.suptitle("Fashion-MNIST: примеры по классам")
plt.tight_layout(); plt.show()

In [ ]:
# Модель AE: encoder -> latent -> decoder
class AE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 256), nn.ReLU(),
            nn.Linear(256, 64),    nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 256),        nn.ReLU(),
            nn.Linear(256, 28*28),     nn.Sigmoid(),
            nn.Unflatten(1, (1, 28, 28)),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


# Проверим размерности
_m = AE(latent_dim=32).to(device)
with torch.no_grad():
    x_hat, z = _m(torch.randn(4, 1, 28, 28).to(device))
print("x_hat:", x_hat.shape, "| z:", z.shape)
print("Параметров:", sum(p.numel() for p in _m.parameters()))

In [ ]:
# Функция обучения + логирование эксперимента
@torch.no_grad()
def eval_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    for xb, _ in loader:
        xb = xb.to(device)
        x_hat, _ = model(xb)
        total += criterion(x_hat, xb).item() * xb.size(0)
    return total / len(loader.dataset)


def train_ae(latent_dim=32, epochs=10, lr=1e-3, verbose=True):
    model = AE(latent_dim=latent_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    history = {"epoch": [], "train_loss": [], "val_loss": [], "time": []}
    run_id = f"ld{latent_dim}_lr{lr}_ep{epochs}"

    for ep in range(epochs):
        t0 = time.time()
        model.train()
        running = 0.0
        for xb, _ in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            x_hat, _ = model(xb)
            loss = criterion(x_hat, xb)
            loss.backward(); optimizer.step()
            running += loss.item() * xb.size(0)
        tr = running / len(train_loader.dataset)
        va = eval_loss(model, test_loader, criterion)
        dt = time.time() - t0

        history["epoch"].append(ep + 1)
        history["train_loss"].append(tr)
        history["val_loss"].append(va)
        history["time"].append(dt)

        if verbose and ((ep + 1) % 2 == 0 or ep == 0):
            print(f"[{run_id}] epoch {ep+1:02d}/{epochs} | train {tr:.4f} | val {va:.4f} | {dt:.1f}s")

    return model, history, run_id

In [ ]:
# Эксперименты: сравним latent_dim = 2, 8, 32
results = []
models = {}
histories = {}

for ld in [2, 8, 32]:
    print(f"\n===== latent_dim = {ld} =====")
    model, hist, run_id = train_ae(latent_dim=ld, epochs=8, lr=1e-3)
    models[ld] = model
    histories[ld] = hist
    results.append({
        "run_id": run_id,
        "latent_dim": ld,
        "epochs": len(hist["epoch"]),
        "final_train_loss": round(hist["train_loss"][-1], 5),
        "final_val_loss":   round(hist["val_loss"][-1], 5),
        "total_time_s":     round(sum(hist["time"]), 1),
        "params":           sum(p.numel() for p in model.parameters()),
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Кривые обучения по всем экспериментам
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ld, hist in histories.items():
    axes[0].plot(hist["epoch"], hist["train_loss"], "--", label=f"ld={ld} train")
    axes[0].plot(hist["epoch"], hist["val_loss"],   "-",  label=f"ld={ld} val")
    axes[1].plot(hist["epoch"], hist["time"],       "-",  label=f"ld={ld}")
axes[0].set_title("Reconstruction MSE"); axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)
axes[1].set_title("Time per epoch (s)"); axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Реконструкции: оригинал vs восстановление (latent_dim=32)
model = models[32]
model.eval()
x_batch, _ = next(iter(test_loader))
x_batch = x_batch.to(device)
with torch.no_grad():
    x_hat, _ = model(x_batch)

n = 8
fig, axes = plt.subplots(2, n, figsize=(16, 4))
for i in range(n):
    axes[0, i].imshow(x_batch[i, 0].cpu(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i, 0].cpu(),  cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_title("Original"); axes[1, 0].set_title("Reconstructed")
plt.suptitle("Реконструкции (latent_dim=32)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Латентное пространство (dim=2): визуально видно, что сеть научилась разделять классы
@torch.no_grad()
def get_latents(model, loader):
    model.eval()
    Z, Y = [], []
    for xb, yb in loader:
        _, z = model(xb.to(device))
        Z.append(z.cpu().numpy())
        Y.append(yb.numpy())
    return np.concatenate(Z), np.concatenate(Y)

Z2, Y2 = get_latents(models[2], test_loader)

plt.figure(figsize=(8, 7))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=Y2, cmap="tab10", s=6, alpha=0.7)
plt.colorbar(sc, ticks=range(10), label="class")
plt.title("Латентное пространство (latent_dim=2)")
plt.xlabel("z1"); plt.ylabel("z2")
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Латентное пространство (dim=32) через PCA и t-SNE
Z32, Y32 = get_latents(models[32], test_loader)

# Возьмём подвыборку, чтобы t-SNE считался быстро
idx = np.random.choice(len(Z32), 3000, replace=False)
Z_sample, Y_sample = Z32[idx], Y32[idx]

Z_pca  = PCA(n_components=2, random_state=42).fit_transform(Z_sample)
Z_tsne = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(Z_sample)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (Z, ttl) in zip(axes, [(Z_pca, "PCA"), (Z_tsne, "t-SNE")]):
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=Y_sample, cmap="tab10", s=6, alpha=0.7)
    ax.set_title(f"Латент (32) — {ttl}")
    plt.colorbar(sc, ax=ax, ticks=range(10))
plt.tight_layout(); plt.show()

## 5. Ретроспектива и следующие шаги

**Что получилось:**
- Полный цикл модельного проекта пройден: charter → данные → модель → эксперименты → визуализации.
- На latent_dim=32 реконструкции визуально похожи на оригиналы, а val loss < 0.02.
- Латентное пространство даже для dim=2 демонстрирует частичное разделение классов.

**Что можно улучшить (Backlog):**
- [ ] Попробовать **VAE** — латент становится распределением, генерация новых объектов.
- [ ] Попробовать **convolutional AE** — качество реконструкции вырастет в разы.
- [ ] Сравнить с **PCA baseline** — сколько компонент нужно для сравнимого MSE.
- [ ] Добавить **early stopping** и **cosine LR schedule**.
- [ ] Логировать эксперименты в **MLflow / W&B** — это следующий уровень зрелости.

**Ретро (что делаем по-другому в следующий раз):**
- Сразу заложить **единый experiment log** (DataFrame), а не собирать метрики вручную.
- Разделять train/val/test **в самом начале**, чтобы не путать оценки.
- Фиксировать **seed** и **версии библиотек** для воспроизводимости.

---

**Итог для студентов:** модельный проект — это не только код модели, но и **артефакты управления**: charter, WBS, Kanban, risk register, experiment log и DoD. Именно они превращают «обучил модель» в проект, который можно повторить и защитить.